# Netflix Audience Rating Classification

This project uses machine learning to predict the audience rating category of Netflix titles.

## Project Workflow

1. Load the dataset
2. Analyze rating categories
3. Create rating categories
4. Select features and target
5. Encode categorical features
6. Split the data into training and testing sets
7. Train Decision Tree and Random Forest models
8. Tune the Random Forest model
9. Evaluate and compare model performance

In [1]:
import pandas as pd

df = pd.read_csv("Dataset.csv")
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
df["rating"].value_counts()

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

In [3]:
def rating_category(rating):
    if rating in ['G', 'TV-G', 'TV-Y', 'TV-Y7']:
        return 'Kids'
    elif rating in ['PG', 'TV-PG', 'PG-13']:
        return 'Teen'
    elif rating in ['R', 'NC-17', 'TV-MA']:
        return 'Adults'
    elif rating in ['NR', 'UR']:
        return 'Not Rated'
    else:
        return 'Other'

df['rating_category'] = df['rating'].apply(rating_category)
df['rating_category'].value_counts()

rating_category
Adults       4007
Other        2163
Teen         1638
Kids          900
Not Rated      82
Name: count, dtype: int64

In [4]:
features = ['type', 'country', 'listed_in', 'release_year', 'duration']
X = df[features]
y = df['rating_category']

X.head()

,type,country,listed_in,release_year,duration
0,Movie,United States,Documentaries,2020,90 min
1,TV Show,France,"Crime TV Shows, International TV Shows, TV Act...",2021,1 Season
2,TV Show,United States,"TV Dramas, TV Horror, TV Mysteries",2021,1 Season
3,Movie,Brazil,"Children & Family Movies, Comedies",2021,91 min
4,Movie,United States,"Dramas, Independent Movies, International Movies",1993,125 min


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['type', 'country', 'listed_in', 'duration']
numeric_features = ['release_year']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)
X = preprocessor.fit_transform(X)

print("Encoded data shape:", X.shape)

Encoded data shape: (8790, 822)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (7032, 822)
Testing data: (1758, 822)


In [7]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Decision Tree training completed")

Decision Tree training completed


In [8]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Decision Tree Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Decision Tree Accuracy: 58.59 %

Classification Report:
              precision    recall  f1-score   support

      Adults       0.64      0.73      0.68       801
        Kids       0.78      0.70      0.74       180
   Not Rated       0.00      0.00      0.00        16
       Other       0.50      0.48      0.49       433
        Teen       0.44      0.34      0.38       328

    accuracy                           0.59      1758
   macro avg       0.47      0.45      0.46      1758
weighted avg       0.58      0.59      0.58      1758



In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest training completed")

Random Forest training completed


In [10]:
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 61.55 %

Classification Report:
              precision    recall  f1-score   support

      Adults       0.64      0.80      0.71       801
        Kids       0.83      0.69      0.76       180
   Not Rated       0.00      0.00      0.00        16
       Other       0.52      0.49      0.50       433
        Teen       0.52      0.33      0.40       328

    accuracy                           0.62      1758
   macro avg       0.50      0.46      0.47      1758
weighted avg       0.60      0.62      0.60      1758



In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", round(grid_search.best_score_ * 100, 2), "%")

Best Parameters: {'max_depth': None, 'n_estimators': 200}
Best Score: 59.63 %


In [12]:
best_model = grid_search.best_estimator_

best_pred = best_model.predict(X_test)

best_accuracy = accuracy_score(y_test, best_pred)

print("Tuned Random Forest Accuracy:", round(best_accuracy * 100, 2), "%")

Tuned Random Forest Accuracy: 62.06 %


In [13]:
print("Decision Tree:", round(accuracy * 100, 2), "%")
print("Random Forest:", round(rf_accuracy * 100, 2), "%")
print("Tuned Random Forest:", round(best_accuracy * 100, 2), "%")

Decision Tree: 58.59 %
Random Forest: 61.55 %
Tuned Random Forest: 62.06 %


In [14]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, best_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[642   3   2 115  39]
 [ 24 126   0  12  18]
 [ 13   0   0   2   1]
 [180   6   0 213  34]
 [135  19   1  63 110]]


In [15]:
print("Tuned Random Forest Classification Report:")
print(classification_report(y_test, best_pred))

Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

      Adults       0.65      0.80      0.72       801
        Kids       0.82      0.70      0.75       180
   Not Rated       0.00      0.00      0.00        16
       Other       0.53      0.49      0.51       433
        Teen       0.54      0.34      0.42       328

    accuracy                           0.62      1758
   macro avg       0.51      0.47      0.48      1758
weighted avg       0.61      0.62      0.61      1758



## Conclusion

In this project, I built a machine learning model to predict the audience rating category of Netflix titles.

I created rating categories and prepared the dataset by encoding categorical features and splitting the data into training and testing sets.

I trained Decision Tree and Random Forest models and compared their performance.

The Decision Tree achieved an accuracy of 58.59%, while the Random Forest achieved 61.55%.

After hyperparameter tuning, the Random Forest achieved the best accuracy of 62.06%.

Based on the test results, the Tuned Random Forest performed the best among the tested models.